> 📓 **Lesson 1.9 — Part 2 of 4: Data Integration — Joining and Reshaping**
>
> This notebook was split out of the original single `eda_advanced.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.9.
>
> Other notebooks in this set: `Part_1_time_series.ipynb`, `Part_3_aggregation_reporting.ipynb`, `Part_4_table_to_decision.ipynb`

# Lesson 1.9: EDA Advanced — Data Wrangling & Analysis

Lesson 1.8 asked *"can I trust this data?"*. This lesson asks the next question:
**what is the pattern, and what should we do about it?**

Clean rows on their own answer nothing. You have to put time on the index, join in the tables that
give the rows meaning, reshape them, and group them. That is the whole job here.

**Structure — the four learning outcomes, in order:**
* **Part 1: Time Series** — *parse* dates, then resample and roll them.
* **Part 2: Data Integration** — *merge* tables, and convert wide ↔ long.
* **Part 3: Aggregation & Reporting** — *aggregate* with `groupby`, `pivot_table`, `crosstab`.
* **Part 4: From Table to Decision** — *apply* all of it to answer the owner's actual question.

**How to read the code cells:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 180 minutes.** One business problem, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Time Series | **Parse** dates; `resample`, `rolling`, `shift` | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Integration | **Merge** tables; `melt` / `pivot` (wide ↔ long) | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Aggregation & Reporting | **Aggregate**: `groupby`, `pivot_table`, `crosstab` | 45 min |
> | **Part 4** | From Table to Decision | **Apply** split-apply-combine to the real question | 20 min |
>
> **The spine:** one business problem — *The Daily Grind*, a four-outlet café chain — and one main
> file, `data/daily_sales.csv`, from start to finish. Small hand-built tables appear alongside as
> *drills*: they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`.


### The four beats of every summary

Lesson 1.8 gave you four beats for every fix: **find it → decide → apply → verify.**
Summarising has its own four, and every table we build today follows them:

| Beat | Ask yourself | |
|---|---|---|
| **1. Question** | What decision does this number serve? | *Renew the Marina Bay lease — yes or no?* |
| **2. Grain** | One row per **what**? | *One row per outlet, per month* |
| **3. Aggregation** | Sum, mean or count — and **why that one**? | *Sum for totals, mean for efficiency* |
| **4. Check** | Does the total still tie back? | *Grouped total == ungrouped total* |

Beat 4 is the one everyone skips. In Part 2 it catches a join that silently deletes $61,310.


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Two toolkits. `pd` and `np` are just short nicknames, so we can type `pd.something`
#    instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np

# 👉 Housekeeping only. Pandas renames a few option strings between versions and shouts
#    about it; this keeps those notices out of our output. Nothing to learn here.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# 👉 The spine. One row per outlet, per day, per part of the day (Morning/Midday/Evening).
#    18 months of trading for a four-outlet café chain, plus a pop-up kiosk.
#    `parse_dates=["date"]` tells pandas: this column is not text, it is a date. More on that in 1.1.
sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])

sales.head()


In [ ]:
# 👉 The 1.8 habit still applies: look before you leap. Shape, types, holes.
print("rows, columns:", sales.shape)
sales.info()


### 🎬 Why this matters — the flat line that hides everything

**The situation.** *The Daily Grind* runs four cafés in Singapore. Revenue has been flat for two
quarters. The Marina Bay lease is up for renewal this month, rent is $9,600, and the owner has to
sign or walk away. She sends you the sales export and asks one question: **what is going on?**

Run the next two cells. The first is the number she already has. The second is the same number,
split by outlet.

> Do not worry about how these two lines work yet — that is Part 1 and Part 3. Just read the output.


In [ ]:
# 👉 Total revenue per quarter for the whole chain -- the headline the owner already has.
#    (`.to_period("Q")` labels each date with its calendar quarter.)
chain_by_quarter = sales.groupby(sales["date"].dt.to_period("Q"))["revenue_sgd"].sum().round(0)

chain_by_quarter


In [ ]:
# 👉 The same revenue, but one column per outlet. Same data. Same period. Different question.
by_outlet = sales.pivot_table(
    index=sales["date"].dt.to_period("Q"),   # down the side: quarter
    columns="outlet_id",                     # across the top: outlet
    values="revenue_sgd",                    # the number in the middle
    aggfunc="sum",                           # how to squash it: add it up
).round(0)

by_outlet


**Read the second table.** OUT-03 falls from about \$138k a quarter to about \$100k. OUT-04 climbs
from about \$96k to about \$131k. One is dying, one is growing, and they move by almost the same
amount — so the chain total barely twitches.

The flat line was never the story. It was **two opposite stories cancelling out**.

No amount of cleaning would have found this. Cleaning gives you rows you can trust; only grouping
turns them into an answer. Three more things you cannot see yet, and will by the end of the session:

1. OUT-03's fall is not a slope. It is a **step**, on one specific week. (Part 1)
2. There is a fifth outlet in this file that does not exist in the outlet list. (Part 2)
3. OUT-03 is still staffed for the revenue it used to make. (Part 3)

Write those three down. We will tick them off.


## Part 2: Data Integration — Joining and Reshaping

**Learning outcome 2:** *Merge multiple DataFrames using SQL-style joins and convert between wide
and long data formats.*

**Goal:** `daily_sales.csv` contains `OUT-03`, not "Marina Bay", and it knows nothing about rent,
seats or staff hours. Those live in other files. Joining is how a row gets its meaning.

⏱️ ~45 min including Group Exercise 2


### 2.1: `merge` — the lookup table

A merge lines up two tables on a shared **key** column. Here the key is `outlet_id`.


In [ ]:
# 👉 The lookup table: one row per outlet, with the attributes the sales file lacks.
outlets = pd.read_csv("../data/outlets.csv", parse_dates=["opened_date"])

outlets


In [ ]:
# 👉 Beat 2 first: choose the grain BEFORE joining. One row per outlet per month.
#    `.reset_index()` turns the grouped index back into ordinary columns so we can merge on them.
monthly = (
    sales.groupby(["outlet_id", "month"])["revenue_sgd"].sum().round(2).reset_index()
)

monthly.head()


In [ ]:
# 👉 The merge. `on="outlet_id"` is the key; `how="left"` means "keep every row on the left,
#    and attach matching columns from the right where they exist".
monthly_named = monthly.merge(outlets, on="outlet_id", how="left")

monthly_named.head(3)


#### The four `how`s, and why the choice is not cosmetic

Our two files disagree about which outlets exist, on purpose:

- `daily_sales.csv` has **OUT-05** — a pop-up kiosk that never made it into the outlet list.
- `outlets.csv` has **OUT-06** (Sentosa Cove) — signed but not yet open, so it has no sales.

Watch what each join does to those two.


In [ ]:
# 👉 The same merge four ways. `.shape[0]` is the row count; the revenue total is beat 4.
for how in ["inner", "left", "right", "outer"]:
    m = monthly.merge(outlets, on="outlet_id", how=how)
    print(
        f"{how:>6}: {m.shape[0]:>3} rows | "
        f"revenue ${m['revenue_sgd'].sum():>12,.0f} | "
        f"outlets seen: {sorted(m['outlet_id'].unique())}"
    )

print(f"\n  raw sales total: ${sales['revenue_sgd'].sum():,.0f}")


> **Tick off finding #2.** The inner join is short by **\$61,310** — the pop-up kiosk's entire
> takings — and it does not warn you. It just quietly returns a smaller number that looks fine.
>
> | `how` | Keeps | Use when |
> |---|---|---|
> | `inner` | only keys in **both** | you need complete attributes on every row |
> | `left` | all of the **left** | the left table is your spine and must not shrink — **the default choice for analysis** |
> | `right` | all of the **right** | rarely; usually clearer written as a `left` the other way round |
> | `outer` | **everything** | reconciling two lists, when the mismatches *are* the finding |
>
> Rule of thumb: use `left` and then check for nulls. Reach for `inner` only when you can say why
> losing rows is correct.


In [ ]:
# 👉 `indicator=True` adds a `_merge` column saying where each row came from.
#    This is the fastest way to see a mismatch instead of guessing at it.
audit = monthly.merge(outlets, on="outlet_id", how="outer", indicator=True)

audit["_merge"].value_counts()


In [ ]:
audit # this is the audit table showing where each row came from in the merge operation.

In [ ]:
# 👉 Which rows are the problem ones? Anything not "both".
audit.loc[audit["_merge"] != "both", ["outlet_id", "month", "revenue_sgd", "outlet_name", "_merge"]].head()


In [ ]:
# 👉 `validate=` makes pandas check the shape of the relationship and raise if it is wrong.
#    "many_to_one": many sales rows, one outlet row. If `outlets` ever gained a duplicate
#    outlet_id, this line would fail loudly instead of silently doubling your revenue.
monthly.merge(outlets, on="outlet_id", how="left", validate="many_to_one").shape


### 2.2: Merging on two keys — the roster

Sometimes one column is not enough to identify a row. The roster is one row per outlet **per week**,
so the key is the pair `(outlet_id, week_start)`.


In [ ]:
# 👉 Weekly staffing per outlet.
roster = pd.read_csv("../data/roster.csv", parse_dates=["week_start"])

roster.head(3)


In [ ]:
# 👉 To join, our sales must be at the same grain: one row per outlet per week, weeks starting
#    Monday. `pd.Grouper` is how you resample INSIDE a groupby of something else.
#    `label="left"` labels each week with its first day, matching the roster's `week_start`.
weekly_sales = (
    sales.groupby(["outlet_id", pd.Grouper(key="date", freq="W-MON", label="left")])["revenue_sgd"]
    .sum()
    .reset_index()
    .rename(columns={"date": "week_start"})
)

weekly_sales.head(3)


In [ ]:
# 👉 Merge on BOTH keys, as a list. Get either key wrong and you get a silent Cartesian mess --
#    which is exactly what `validate="one_to_one"` is there to prevent.
staffed = weekly_sales.merge(roster, on=["outlet_id", "week_start"], how="inner", validate="one_to_one")

# 👉 The efficiency question the owner actually cares about: dollars earned per staff hour paid.
staffed["rev_per_staff_hour"] = (staffed["revenue_sgd"] / staffed["staff_hours"]).round(2)

staffed.head(3)


> **Watch the dtypes when you merge.** `week_start` had to be a real date on *both* sides. Merging
> a text `"2024-01-01"` against a `Timestamp("2024-01-01")` matches nothing, and pandas reports
> zero matches rather than an error. If a merge returns suspiciously few rows, check `.dtypes` first.


### 2.3: Wide → long with `melt`

The targets file is laid out the way a manager types it into Excel: one row per outlet, **one
column per month**. Convenient for reading, useless for joining — "month" is not a column, it is
a set of headers.


In [ ]:
# 👉 Look at the shape of the problem first.
targets_wide = pd.read_csv("../data/targets_wide.csv")

targets_wide.iloc[:, :6]


In [ ]:
# 👉 `melt` unpivots. `id_vars` are the columns to KEEP as they are; everything else gets folded
#    down into two new columns: one holding the old header, one holding the value.
targets = targets_wide.melt(
    id_vars="outlet_id",        # keep this as a column
    var_name="month",           # the old column headers land here
    value_name="target_sgd",    # the numbers land here
)

print(f"{targets_wide.shape} wide  ->  {targets.shape} long")
targets.head()


In [ ]:
# 👉 Now `month` is a real column, so it can be a merge key. Both sides must be the same type,
#    so convert our Period month to text to match the target sheet's "2024-01" strings.
monthly["month_str"] = monthly["month"].astype(str)

performance = monthly.merge(targets, left_on=["outlet_id", "month_str"], right_on=["outlet_id", "month"], how="left")
performance["variance_pct"] = ((performance["revenue_sgd"] / performance["target_sgd"] - 1) * 100).round(1)

performance[["outlet_id", "month_str", "revenue_sgd", "target_sgd", "variance_pct"]].tail(8)


> **The variance column is the point.** Every month of it was invisible while the targets sat in a
> wide sheet. Reshaping is not tidiness for its own sake — it is what makes a comparison possible.
>
> Notice the last three rows: the pop-up kiosk has a `NaN` target, because nobody set it one. That
> is the honest answer. If you had used `.fillna(0)` here, the kiosk would show a variance of
> "+∞% above target" — a number that is arithmetically fine and completely false.


### 2.4: Long → wide with `pivot`

`pivot` is `melt` run backwards: it takes a column of labels and spreads it across the top.
Long format is for computers; wide format is for people. Reshape at the last minute, for reading.


In [ ]:
# 👉 One row per month, one column per outlet. `pivot` needs three things: what goes down the
#    side (index), what goes across the top (columns), and what fills the middle (values).
wide_view = monthly.pivot(index="month_str", columns="outlet_id", values="revenue_sgd").round(0)

wide_view.tail(6)


In [ ]:
# 👉 `pivot` fails if a single (index, columns) cell would hold more than one value -- it has
#    no instruction for what to do with the second one. Here, two rows land in the same cell:
clash = pd.DataFrame({
    "month": ["2025-06", "2025-06"],
    "outlet_id": ["OUT-01", "OUT-01"],
    "revenue_sgd": [100, 200],
})
display(clash) # show the clash table, which has two rows for the same month and outlet.
try:
    clash.pivot(index="month", columns="outlet_id", values="revenue_sgd")
except ValueError as e:
    print("ValueError:", e)


In [ ]:
# 👉 `pivot_table` is the same operation PLUS an aggregation, so duplicates are fine --
#    you tell it how to combine them. That is the only real difference between the two.
clash.pivot_table(index="month", columns="outlet_id", values="revenue_sgd", aggfunc="sum")


### 🛠️ Group Exercise 2 — Integration (8 min)

Using `staffed` from section 2.2, find the average `rev_per_staff_hour` for each outlet across the whole period. Which outlet earns the least per staff hour?

*Expected:* five rows, and `OUT-03` is clearly the worst of the four permanent outlets.

---

## ✅ Sample Solution

Try the exercise yourself first — this is *a* solution, not *the* solution. If your code reaches the same answer a different way, it is right.

**Average `rev_per_staff_hour` per outlet.**

In [ ]:
# 👉 The straightforward reading: average the weekly ratios.
staffed.groupby("outlet_id")["rev_per_staff_hour"].mean().round(2).sort_values()

In [ ]:
# 👉 The more defensible version: total revenue / total hours, so a quiet week does not
#    count as much as a busy one. Same ranking here -- worth checking, not assuming.
totals = staffed.groupby("outlet_id").agg(
    revenue=("revenue_sgd", "sum"),
    staff_hours=("staff_hours", "sum"),
)

(totals["revenue"] / totals["staff_hours"]).round(2).sort_values()

Five rows, and `OUT-03` (Marina Bay) is bottom at ~\$23.3 per staff hour against ~\$26–27 for the other three permanent outlets. `OUT-05`, the kiosk, sits mid-table on three months of data — do not rank it against 18 months of trading. This is finding #3, reached from the weekly table instead of the quarterly one.

Note the two methods answer slightly different questions. The mean of ratios treats every week equally; revenue ÷ hours weights each week by its size. When the two disagree, the disagreement is itself the finding.

---

# ☕ Break — 10 minutes

**Where we are:** the tables are joined and reshaped. Every row now knows its outlet name, region,
rent and staffing, and the targets are joinable.

**Next up:** Part 3 — the summarising engine. `groupby`, `pivot_table` and `crosstab`, and the two
tables that decide the lease.


📂 **Open** `Part_3_aggregation_reporting.ipynb` to continue.